In [ ]:
%load_ext autoreload
%autoreload 2
import pyarrow as pa
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_data
from src.vector_store import create_chroma_collection,add_embeddings
from src.preprocessing import create_stratified_sample,chunk_dataset,chunk_text
from src.embedding import (load_embedding_model,generate_embeddings)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
filtered_complaints = load_data("../data/processed/filtered_complaints.csv")

Dataset loaded successfully.
Rows: 549585
Columns: 20


# Stratified Sampling

In [4]:
sample_df = create_stratified_sample(filtered_complaints, sample_size=10000)

print(sample_df.shape)

(10000, 20)


In [5]:
distribution = (
    sample_df["Product Category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(distribution)

Product Category
Credit Card        34.45
Savings Account    25.53
Personal Loan      22.06
Money Transfer     17.96
Name: proportion, dtype: float64


In [6]:
sample_df["Product Category"].value_counts()

Product Category
Credit Card        3445
Savings Account    2553
Personal Loan      2206
Money Transfer     1796
Name: count, dtype: int64

In [7]:
sample_df.to_csv(
    "../data/processed/filtered_complaints_sample.csv",
    index=False
)

# Chunking

In [8]:
chunked_df = chunk_dataset(
    sample_df,
    chunk_size=500,
    chunk_overlap=50
)

print(chunked_df.shape)

chunked_df.head()

(29099, 23)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,Product Category,chunk,chunk_index,total_chunks
0,2025-02-12,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,my account was closed from xxxx and at the tim...,NaN,"Block, Inc.",TX,77583,...,2025-02-12,Closed with explanation,Yes,NaN,12072607,180,Savings Account,my account was closed from xxxx and at the tim...,0,2
0,2025-02-12,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,my account was closed from xxxx and at the tim...,NaN,"Block, Inc.",TX,77583,...,2025-02-12,Closed with explanation,Yes,NaN,12072607,180,Savings Account,can trace my funds and return them to me i hav...,1,2
1,2025-01-20,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,i am filing a complaint against cash app block...,NaN,"Block, Inc.",CO,80205,...,2025-01-20,Closed with explanation,Yes,NaN,11614309,102,Money Transfer,i am filing a complaint against cash app block...,0,2
1,2025-01-20,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,i am filing a complaint against cash app block...,NaN,"Block, Inc.",CO,80205,...,2025-01-20,Closed with explanation,Yes,NaN,11614309,102,Money Transfer,under the electronic fund transfer act efta an...,1,2
2,2025-03-13,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,Unauthorized transactions or other transaction...,NaN,cash app respond is insane said they couldnt g...,NaN,"Block, Inc.",VA,237XX,...,2025-03-13,Closed with explanation,Yes,NaN,12457083,12,Money Transfer,cash app respond is insane said they couldnt g...,0,1


In [10]:
configs = [
    (300,30),
    (500,50),
    (700,100)
]

for chunk_size, overlap in configs:

    chunked = chunk_dataset(
        sample_df,
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    print(f"\nChunk size: {chunk_size}")
    print(f"Overlap: {overlap}")
    print(f"Total chunks: {len(chunked)}")

    avg_length = chunked["chunk"].str.len().mean()

    print(f"Average chunk length: {avg_length:.1f}")

KeyboardInterrupt: 

In [13]:
text = sample_df.iloc[0]["Consumer complaint narrative"]

for size, overlap in [(300,30),(500,50),(600,70),(700,100)]:

    chunks = chunk_text(
        text,
        chunk_size=size,
        chunk_overlap=overlap
    )

    print("="*80)
    print(size, overlap)

    for i,c in enumerate(chunks[:3]):
        print(f"\nChunk {i+1}\n")
        print(c)

300 30

Chunk 1

my account was closed from xxxx and at the time i had several ach items that were supposed to be returned to my cashapp account that never were i wanted to reach back out to see where my funds actually went before i file my complaint with the xxxx and the fdic in regards to my missing funds i have

Chunk 2

to my missing funds i have been waiting on over 2000 00 i contacted the bank and they stated the money was sent back to cashapp so i contacted cashapp with the trace number so they can trace my funds and return them to me i have been more than patient trying to track down my money and no one wants

Chunk 3

my money and no one wants to be held accountable when my funds were returned to the account cashapp could have simply mailed me a check but they didn t do that and now my money seems to be lost in the wind the has caused me financial distress and i need to know where my money is because it has been
500 50

Chunk 1

my account was closed from xxxx and at the time 

# Generate Embeddings

In [11]:

model = load_embedding_model()

chunked_df = generate_embeddings(
    chunked_df,
    model
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/455 [00:00<?, ?it/s]

In [12]:
chunked_df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,...,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count,Product Category,chunk,chunk_index,total_chunks,embedding
0,2025-02-12,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,my account was closed from xxxx and at the tim...,NaN,"Block, Inc.",TX,77583,...,Closed with explanation,Yes,NaN,12072607,180,Savings Account,my account was closed from xxxx and at the tim...,0,2,"[0.013290705159306526, -0.012922031804919243, ..."
0,2025-02-12,Checking or savings account,Other banking product or service,Managing an account,Funds not handled or disbursed as instructed,my account was closed from xxxx and at the tim...,NaN,"Block, Inc.",TX,77583,...,Closed with explanation,Yes,NaN,12072607,180,Savings Account,can trace my funds and return them to me i hav...,1,2,"[0.030819974839687347, -0.01960556022822857, -..."
1,2025-01-20,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,i am filing a complaint against cash app block...,NaN,"Block, Inc.",CO,80205,...,Closed with explanation,Yes,NaN,11614309,102,Money Transfer,i am filing a complaint against cash app block...,0,2,"[-0.12191501259803772, 0.054789312183856964, 0..."
1,2025-01-20,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,i am filing a complaint against cash app block...,NaN,"Block, Inc.",CO,80205,...,Closed with explanation,Yes,NaN,11614309,102,Money Transfer,under the electronic fund transfer act efta an...,1,2,"[0.01816040836274624, 0.02550520747900009, -0...."
2,2025-03-13,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,Unauthorized transactions or other transaction...,NaN,cash app respond is insane said they couldnt g...,NaN,"Block, Inc.",VA,237XX,...,Closed with explanation,Yes,NaN,12457083,12,Money Transfer,cash app respond is insane said they couldnt g...,0,1,"[-0.09607723355293274, 0.017643800005316734, 0..."


In [13]:
embedding = chunked_df.iloc[0]["embedding"]

print(len(embedding))

384


In [14]:
chunked_df.to_parquet(
    "../data/processed/complaint_embeddings.parquet",
    index=False
)

In [19]:
print(chunked_df.index.is_unique)

False


In [20]:
chunked_df = chunked_df.reset_index(drop=True)

# Build Vector Database

In [21]:
collection = create_chroma_collection()

add_embeddings(
    collection,
    chunked_df
)

Indexing Chunks:   0%|          | 0/6 [00:00<?, ?it/s]


Successfully indexed 29,099 chunks.


Collection(name=complaints)

In [23]:
collection.peek()

{'ids': ['chunk_0',
  'chunk_1',
  'chunk_2',
  'chunk_3',
  'chunk_4',
  'chunk_5',
  'chunk_6',
  'chunk_7',
  'chunk_8',
  'chunk_9'],
 'embeddings': array([[ 0.01329071, -0.01292203, -0.0542267 , ..., -0.05230036,
         -0.00762858,  0.00726825],
        [ 0.03081997, -0.01960556, -0.05421557, ..., -0.04685511,
          0.05140719,  0.02350499],
        [-0.12191501,  0.05478931,  0.02025292, ..., -0.03437499,
          0.06055311,  0.05405391],
        ...,
        [ 0.05693175,  0.03280631, -0.00580777, ..., -0.03125641,
          0.03252798, -0.05417583],
        [ 0.03694901, -0.07664038,  0.00065599, ..., -0.05211463,
         -0.11491901, -0.04376294],
        [-0.01256569, -0.00941585,  0.04053326, ..., -0.0736514 ,
         -0.01445731, -0.04286427]], shape=(10, 384)),
 'documents': ['my account was closed from xxxx and at the time i had several ach items that were supposed to be returned to my cashapp account that never were i wanted to reach back out to see where my f